In [18]:
# BIBLIOTECAS UTILIZADAS

import pandas as pd
from statsmodels.tsa.stattools import adfuller
from prophet import Prophet
from pmdarima import auto_arima  
import plotly.graph_objects as go
from sklearn.metrics import r2_score
import plotly.graph_objects as go
from pathlib import Path

In [19]:
# CARREGANDO O DATASET DE CLUSTERING

csv_path = Path("..") / "model" / "SUPERSTORE_CLUSTERING.csv"
df_clustering = pd.read_csv(csv_path)

In [20]:
# TRATAMENTO DE DADOS PARA SÉRIES TEMPORAIS

df_clustering['ORDER_DATE'] = pd.to_datetime(df_clustering['ORDER_DATE'])
df_temporal = df_clustering.groupby([df_clustering['ORDER_DATE'].dt.to_period('M'), 'CLUSTER'])['PROFIT'].sum().reset_index()
df_temporal['ORDER_DATE'] = df_temporal['ORDER_DATE'].dt.to_timestamp() 
df_temporal = df_temporal.sort_values('ORDER_DATE').reset_index(drop=True)

In [21]:
# SEPARAÇÃO DOS DATAFRAMES POR CLUSTER E CRIAÇÃO DE DICIONÁRIO

df_temporal_cluster_0 = df_temporal[df_temporal['CLUSTER'] == 0].copy()
df_temporal_cluster_1 = df_temporal[df_temporal['CLUSTER'] == 1].copy()
df_temporal_cluster_2 = df_temporal[df_temporal['CLUSTER'] == 2].copy()

df_temporal_dict = {0: df_temporal_cluster_0, 1: df_temporal_cluster_1, 2: df_temporal_cluster_2}

In [22]:
# VISUALIZAÇÃO DAS SÉRIES TEMPORAIS DE LUCRO POR CLUSTER

for i, data in df_temporal_dict.items():
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=data['ORDER_DATE'],
            y=data['PROFIT'],
            mode='lines+markers',
            name=f'Cluster {i}'
        )
    )
    fig.update_layout(
        title=f"Lucro - Cluster {i}",
        xaxis_title="Data do Pedido",
        yaxis_title="Lucro",
    )
    fig.show()

Percebe-se uma mudança evidente a partir de 2020, para conseguir resultados mais acertivos, vamos eliminar os dados antes da segunda metade de 2020.

In [ ]:
# ELIMINAÇÃO DE PERÍODOS ANTERIORES A JULHO DE 2020 

for i in df_temporal_dict:
    df_temporal_dict[i] = df_temporal_dict[i][df_temporal_dict[i]['ORDER_DATE'] >= '2020-07-01']

In [ ]:
# TRATAMENTO PRÉ-ARIMA: IDENTIFICAÇÃO DE ESTACIONARIEDADE E DIFERENCIAÇÃO

for i, data in df_temporal_dict.items():
    print(f"Cluster {i}:")
    
    ts = data['PROFIT'].copy()
    
    while True:
        result = adfuller(ts)
        print(f'  ADF Statistic: {result[0]:.4f}')
        print(f'  p-value: {result[1]:.4f}')
        
        if result[1] < 0.05:
            print("  Estacionária\n")
            break
        else:
            print("  Não estacionária, diferenciando...\n")
            ts = ts.diff().dropna()

Cluster 0:
  ADF Statistic: -1.3520
  p-value: 0.6050
  Não estacionária, diferenciando...

  ADF Statistic: -0.6349
  p-value: 0.8629
  Não estacionária, diferenciando...

  ADF Statistic: -0.9644
  p-value: 0.7660
  Não estacionária, diferenciando...

  ADF Statistic: -6.8015
  p-value: 0.0000
  Estacionária

Cluster 1:
  ADF Statistic: -1.0493
  p-value: 0.7349
  Não estacionária, diferenciando...

  ADF Statistic: -2.2270
  p-value: 0.1966
  Não estacionária, diferenciando...

  ADF Statistic: -3.1491
  p-value: 0.0231
  Estacionária

Cluster 2:
  ADF Statistic: -4.6903
  p-value: 0.0001
  Estacionária



In [ ]:
# APLICAÇÃO DO MODELO ARIMA AUTOMÁTICO PARA CADA CLUSTER

arima_results = {}

for i, data in df_temporal_dict.items():
    ts = data.set_index('ORDER_DATE')['PROFIT']
    ts = ts.asfreq('MS', fill_value=0)

    train = ts[:int(0.8 * len(ts))]
    test = ts[int(0.8 * len(ts)):]

    print(f"--- Cluster {i} ---")
    auto_model = auto_arima(train, seasonal=False, trace=True, verbose=0)

    forecast = auto_model.predict(n_periods=len(test))
    forecast_series = pd.Series(forecast, index=test.index)

    arima_results[i] = {'train': train, 'test': test, 'forecast': forecast_series, 'model': auto_model}
    print('\n')

--- Cluster 0 ---
Performing stepwise search to minimize aic
 ARIMA(2,1,2)(0,0,0)[0] intercept   : AIC=inf, Time=0.05 sec
 ARIMA(0,1,0)(0,0,0)[0] intercept   : AIC=292.112, Time=0.01 sec
 ARIMA(1,1,0)(0,0,0)[0] intercept   : AIC=291.245, Time=0.06 sec
 ARIMA(0,1,1)(0,0,0)[0] intercept   : AIC=inf, Time=0.04 sec
 ARIMA(0,1,0)(0,0,0)[0]             : AIC=290.381, Time=0.01 sec
 ARIMA(1,1,1)(0,0,0)[0] intercept   : AIC=inf, Time=0.07 sec

Best model:  ARIMA(0,1,0)(0,0,0)[0]          
Total fit time: 0.237 seconds


--- Cluster 1 ---
Performing stepwise search to minimize aic
 ARIMA(2,1,2)(0,0,0)[0] intercept   : AIC=inf, Time=0.10 sec
 ARIMA(0,1,0)(0,0,0)[0] intercept   : AIC=304.413, Time=0.01 sec
 ARIMA(1,1,0)(0,0,0)[0] intercept   : AIC=295.100, Time=0.02 sec
 ARIMA(0,1,1)(0,0,0)[0] intercept   : AIC=inf, Time=0.02 sec
 ARIMA(0,1,0)(0,0,0)[0]             : AIC=302.506, Time=0.01 sec
 ARIMA(2,1,0)(0,0,0)[0] intercept   : AIC=296.770, Time=0.03 sec
 ARIMA(1,1,1)(0,0,0)[0] intercept   : A

In [ ]:
# APRESENTAÇÃO DOS RESULTADOS DO ARIMA PARA CADA CLUSTER

forecast_months = 3

for i, data in df_temporal_dict.items():
    ts = data.copy()

    model = Prophet()
    model.fit(ts.rename(columns={'ORDER_DATE': 'ds', 'PROFIT': 'y'}))
    future = model.make_future_dataframe(periods=forecast_months, freq='MS')
    forecast = model.predict(future)

    forecast_in_sample = forecast[forecast['ds'].isin(ts['ORDER_DATE'])]
    y_true = ts.set_index('ORDER_DATE').loc[forecast_in_sample['ds']]['PROFIT'].values
    y_pred = forecast_in_sample['yhat'].values

    r2 = r2_score(y_true, y_pred)

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=ts['ORDER_DATE'], y=ts['PROFIT'], mode='lines', name='Observado', line=dict(color='blue')))
    fig.add_trace(go.Scatter(x=forecast['ds'], y=forecast['yhat'], mode='lines', name='Previsão', line=dict(color='red', dash='dash')))
    fig.add_trace(go.Scatter(x=forecast['ds'], y=forecast['yhat_upper'], mode='lines', name='Upper', line=dict(width=0), showlegend=False))
    fig.add_trace(go.Scatter(x=forecast['ds'], y=forecast['yhat_lower'], mode='lines', name='Intervalo de Confiança', fill='tonexty', fillcolor='rgba(255,0,0,0.15)', line=dict(width=0)))
    
    mae = (y_true - y_pred).mean()
    mse = ((y_true - y_pred) ** 2).mean()

    fig.update_layout(
        title=f"Prophet Forecast - Cluster {i} ({forecast_months} meses) | R² = {r2:.3f} | MAE = {mae:.3f} | MSE = {mse:.3f}",
        xaxis_title="Order Date",
        yaxis_title="Profit",
        height=500
    )
    fig.show()

18:14:22 - cmdstanpy - INFO - Chain [1] start processing
18:14:22 - cmdstanpy - INFO - Chain [1] done processing


18:14:22 - cmdstanpy - INFO - Chain [1] start processing
18:14:22 - cmdstanpy - INFO - Chain [1] done processing


18:14:22 - cmdstanpy - INFO - Chain [1] start processing
18:14:22 - cmdstanpy - INFO - Chain [1] done processing
